# mulenet

Data: IBM Transactions for AML (`ealtman2019/ibm-transactions-for-anti-money-laundering-aml`), HI-Small split.

- **Kaggle:** the dataset is attached via `kernel-metadata.json` and read from `/kaggle/input/...` (never downloaded).
- **Local:** files are read from `data/raw/` (fetched once with the Kaggle CLI).

In [ ]:
import os
from pathlib import Path

REQUIRED_FILES = ["HI-Small_Trans.csv", "HI-Small_Patterns.txt"]

ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or Path("/kaggle/input").is_dir()


def find_data_dir() -> Path:
    if ON_KAGGLE:
        # Mount layout under /kaggle/input varies, so locate the file rather than hardcode the path.
        hits = sorted(Path("/kaggle/input").rglob(REQUIRED_FILES[0]))
        if not hits:
            raise FileNotFoundError("Dataset not attached: add it to dataset_sources in kernel-metadata.json")
        return hits[0].parent
    for candidate in (Path("data/raw"), Path("../data/raw")):
        if (candidate / REQUIRED_FILES[0]).is_file():
            return candidate.resolve()
    raise FileNotFoundError("data/raw/ not found: download the files with the Kaggle CLI first (see prompt.md)")


DATA_DIR = find_data_dir()
missing = [f for f in REQUIRED_FILES if not (DATA_DIR / f).is_file()]
assert not missing, f"missing in {DATA_DIR}: {missing}"
TRANS_PATH = DATA_DIR / "HI-Small_Trans.csv"
PATTERNS_PATH = DATA_DIR / "HI-Small_Patterns.txt"
# Optional robustness-check split; same directory when present.
LI_TRANS_PATH = DATA_DIR / "LI-Small_Trans.csv"
LI_PATTERNS_PATH = DATA_DIR / "LI-Small_Patterns.txt"
HAS_LI = LI_TRANS_PATH.is_file() and LI_PATTERNS_PATH.is_file()
print(f"environment: {'kaggle' if ON_KAGGLE else 'local'}")
print(f"data dir:    {DATA_DIR}")
print(f"LI-Small:    {'available' if HAS_LI else 'not present'}")

## Load

In [ ]:
import pandas as pd

# The raw header names both account columns "Account", so we supply our own names.
COLUMNS = [
    "timestamp", "from_bank", "from_account", "to_bank", "to_account",
    "amount_received", "receiving_currency", "amount_paid", "payment_currency",
    "payment_format", "is_laundering",
]
# Bank codes have leading zeros ("010"), so keep ids as strings.
DTYPES = {
    "from_bank": str, "from_account": str, "to_bank": str, "to_account": str,
    "receiving_currency": str, "payment_currency": str, "payment_format": str,
    "amount_received": "float64", "amount_paid": "float64", "is_laundering": "int8",
}
TIME_FORMAT = "%Y/%m/%d %H:%M"


def load_transactions(path) -> pd.DataFrame:
    df = pd.read_csv(path, header=0, names=COLUMNS, dtype=DTYPES)
    df["timestamp"] = pd.to_datetime(df["timestamp"], format=TIME_FORMAT)
    return df


def load_patterns(path) -> tuple[pd.DataFrame, int, int]:
    """Parse the patterns file into one row per listed transaction.

    Returns (rows, number of BEGIN lines, number of END lines). Each row carries
    `attempt` (0-based BEGIN index) and `pattern` (e.g. "FAN-OUT").
    """
    rows, begins, ends, pattern = [], 0, 0, None
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith("BEGIN"):
                # "BEGIN LAUNDERING ATTEMPT - FAN-OUT:  Max 16-degree Fan-Out" -> "FAN-OUT"
                pattern = line.split(" - ", 1)[1].split(":", 1)[0].strip()
                begins += 1
            elif line.startswith("END"):
                pattern = None
                ends += 1
            else:
                if pattern is None:
                    raise ValueError(f"transaction outside a BEGIN/END block: {line!r}")
                rows.append([begins - 1, pattern, *line.split(",")])
    df = pd.DataFrame(rows, columns=["attempt", "pattern", *COLUMNS])
    df = df.astype({k: v for k, v in DTYPES.items()})
    df["timestamp"] = pd.to_datetime(df["timestamp"], format=TIME_FORMAT)
    return df, begins, ends

## Data checks

Expected numbers come from `summary.md` (computed by `scripts/verify_data.py`). LI-Small is checked only when its files are present.

In [ ]:
EXPECTED = {
    "HI-Small": {"rows": 5_078_345, "laundering": 5_177, "attempts": 370},
    "LI-Small": {"rows": 6_924_049, "laundering": 3_565, "attempts": 117},
}
KEY = [c for c in COLUMNS if c != "is_laundering"]


def check_split(prefix: str) -> dict:
    trans = load_transactions(DATA_DIR / f"{prefix}_Trans.csv")
    patterns, begins, ends = load_patterns(DATA_DIR / f"{prefix}_Patterns.txt")
    exp = EXPECTED[prefix]

    # Counts match what was recorded at download time.
    assert len(trans) == exp["rows"], (len(trans), exp["rows"])
    assert int(trans["is_laundering"].sum()) == exp["laundering"]
    assert begins == ends == exp["attempts"], (begins, ends)

    # Integrity of the transactions table.
    assert not trans.isna().any().any(), trans.isna().sum()[lambda s: s > 0]
    assert set(trans["is_laundering"].unique()) <= {0, 1}
    assert (trans["amount_received"] >= 0).all() and (trans["amount_paid"] >= 0).all()

    # Integrity of the patterns file.
    assert (patterns["is_laundering"] == 1).all(), "patterns file lists a non-laundering transaction"
    assert patterns.groupby("attempt").size().min() > 0

    # Every transaction in the patterns file should exist in the CSV, labelled 1.
    labels = trans.drop_duplicates(KEY)[KEY + ["is_laundering"]]
    joined = patterns[KEY].merge(labels, on=KEY, how="left")
    found = joined["is_laundering"].notna()
    assert (joined.loc[found, "is_laundering"] == 1).all(), "pattern transaction labelled 0 in CSV"

    in_patterns = trans.merge(patterns[KEY].drop_duplicates(), on=KEY, how="inner")
    result = {
        "split": prefix,
        "rows": len(trans),
        "laundering": int(trans["is_laundering"].sum()),
        "attempts": begins,
        "pattern_txns": len(patterns),
        "pattern_txns_found_in_csv": int(found.sum()),
        "laundering_rows_covered_by_patterns": int(in_patterns["is_laundering"].sum()),
        "duplicate_rows": int(trans.duplicated().sum()),
        "time_sorted": bool(trans["timestamp"].is_monotonic_increasing),
        "first": trans["timestamp"].min(),
        "last": trans["timestamp"].max(),
    }
    del trans, patterns, labels, joined, in_patterns
    return result


splits = ["HI-Small"] + (["LI-Small"] if HAS_LI else [])
results = pd.DataFrame([check_split(s) for s in splits]).set_index("split").T
print("all data checks passed")
results